In [0]:
import requests
import json
from pyspark.sql.functions import current_date, date_format, current_timestamp

SCHEMA_DESTINY = "BRONZE"
TB_NAME = "TB_BITCOIN"
TB_DESTINY = f"{SCHEMA_DESTINY}.{TB_NAME}"

In [0]:

API_KEY = "AFSD5SKCOEDWSR7GEU6DZ"

url = "https://www.alphavantage.co/query"
params = {
    "function": "DIGITAL_CURRENCY_DAILY",
    "symbol": "BTC",
    "market": "USD",
    "apikey": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

series = data["Time Series (Digital Currency Daily)"]

# Monta lista de registros mantendo as chaves originais da API
registros = []
for data_dia, valores in series.items():
    linha = {"data": data_dia}
    for chave, valor in valores.items():
        nova_chave = chave.split(". ", 1)[1]  # remove "1. ", "2. ", etc.
        linha[nova_chave] = valor
    registros.append(linha)

# Cria o DataFrame Spark direto, sem tratamento
df_bitcoin = spark.createDataFrame(registros)

df_bitcoin = (df_bitcoin.withColumn("DT_HR_LOAD", current_timestamp())
              .withColumn("DT_LOAD", current_date())
              .withColumn("HR_LOAD", date_format(current_timestamp(), "HH:mm:ss")))

df_bitcoin.write.format('delta').mode('overwrite').saveAsTable(TB_DESTINY)